## Data Ingestion


In [3]:
from langchain_community.document_loaders import PyMuPDFLoader,PyPDFLoader
from langchain.document_loaders import DirectoryLoader
# Load PDF documenta
dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",# pattern to match files
    loader_cls=PyMuPDFLoader, # loader class to use
    show_progress=False# show progress bar

)
pdf_documents=dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'Skia/PDF m140 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\mcp project.pdf', 'file_path': '..\\data\\pdf\\mcp project.pdf', 'total_pages': 7, 'format': 'PDF 1.4', 'title': 'Untitled document', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content="Project Documentation: Multi-Server \nMCP Agent for Math and Weather \n1. Project Overview \nThis project demonstrates a robust multi-server agent architecture using Python. The system \nfeatures a central client that orchestrates tasks by communicating with two separate, \nspecialized tool servers: \n1.\u200b A Math Server that performs basic arithmetic operations, communicating via stdio. \n2.\u200b A Weather Server that fetches live weather data from an external API, \ncommunicating over http. \nThe client uses a Large Language Model (LLM) from Groq, integrated into a LangGraph \nReAct age

In [ ]:
##chunking the docu

from langchain_community.document_loaders import PyMuPDFLoader  # or PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from typing import List
from langchain.schema import Document

def load_and_chunk_pdf(
    file_path: str,
    chunk_size: int = 1000,
    chunk_overlap: int = 200
) -> List[Document]:
    """
    Load a PDF and split it into smaller text chunks for RAG.
    
    Args:
        file_path (str): Path to the PDF file
        chunk_size (int): Maximum number of characters per chunk
        chunk_overlap (int): Overlap between chunks to preserve context
    
    Returns:
        List[Document]: Chunked documents
    """
    # Step 1: Load PDF
    loader = PyMuPDFLoader(file_path)  # Fast + supports images metadata
    documents = loader.load()

    # Step 2: Split into chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    chunks = text_splitter.split_documents(documents)

    print(f"✅ Loaded {len(documents)} pages")
    print(f"✅ Split into {len(chunks)} chunks")
    return chunks


## embedding and vector


In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\mayank manjhi\Dropbox\rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class EmbeddingManager:
    "Handels doc embedding gnerations using sentence transformers "
    
    def __init__(self,model_name: str="all-MiniLM-L6-v2"):
        """"initialize the embedding mNAGER
        ARGS:
            model_name: HUGGINGFACE model namr for sentence embedding
        """
        self.model_name=model_name
        self.model=None
        self._load_model()
    
    def _load_model(self):
        ## load the sentence transformer model
        try:
            print(f"loading embedding model:{self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"model loaded successfully.embedding dimension :{self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model:{self.model_name}:{e}")
            raise
    
    def genrate_embedding(self,texts:List[str])->np.ndarray:
        """Generate embeddings for a list of texts
        ARGS:
            texts: List of texts to embed

        RETURNS:
            np.ndarray: Array of embeddings with shape (len(texts), embedding_dim)"""
        if not self.model:
            raise ValueError("Model not loaded") 
        print(f"Generating embeddings for {len(texts)} texts..")
        embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"Genrated embeddings with shape:{np.array(embeddings).shape}")
        return embeddings
    
## initializing the embedding manager
embedding_manager=EmbeddingManager()
embedding_manager
    
    



loading embedding model:all-MiniLM-L6-v2


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model loaded successfully.embedding dimension :384


In [21]:
from typing import List,Dict,Any
import os
import numpy as np
import chromadb
import uuid 
## vectore store
class VectorStore:
    "manages document embbeding in chromadb vector store"
    def __init__(self,collection_name: str = "pdf_ocuments", persist_directory: str="../data/vector_strore"):
        """initialize the vector store
        ARGS:
            collection_name: Name of the collection to use
            persist_directory: Directory to persist the vector store
        """
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialize_store()
    
    def _initialize_store(self):
        """initialize the chroma db and vector collection"""
        try:
            # create persistant chroma db client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)

            # get or create collection
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "pdf docu embbeding for RAg"}

            )
            print(f" vector store initialized , collection:{self.collection_name}")
            print(f" existing docu in collection:{self.collection.count()}")
        
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    def add_documents(self,documents:List[Any],embeddings: np.ndarray):
        """add documents and their embeddings to the vector store
        
        Args:
            documents: List of langchain docu
            embeddings: corresponding embedding for th docu"""
        
        if len(documents)!= len(embeddings):
            raise ValueError("Number of documents and embeddings must be the same")
       
        print(f"Adding {len(documents)} documents to the vector store...")
        
        ##prepare data for chromadb
        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]
        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):
            ##genrate unique id for each socu
            doc_id=f"doc_{uuid.uuid4().hex[:8]}{i}"
            ids.append(doc_id)

            ## prepare metadata
            metadata=dict(doc.metadata)
            metadata["doc_index"]=i
            metadata["contnet_length"]=len(doc.page_content)
            metadatas.append(metadata)

            ## STORE TEXT
            documents_text.append(doc.page_content)

            #embedding
            embeddings_list.append(embedding.tolist())

        ## add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to the vector store")
            print(f"Total documents in collection:{self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store:{e}")
            raise
"""
vectorstore=VectorStore()
vectorstore"""
vectorstore = VectorStore()
vectorstore



 vector store initialized , collection:pdf_ocuments
 existing docu in collection:0


In [26]:
docs = load_and_chunk_pdf("../data/pdf/mcp project.pdf",chunk_size=800,chunk_overlap=100)

# Each doc has text + metadata
print(docs[0].page_content[:200])   # first 200 chars of first chunk
print(docs[0].metadata)


✅ Loaded 7 pages
✅ Split into 15 chunks
Project Documentation: Multi-Server 
MCP Agent for Math and Weather 
1. Project Overview 
This project demonstrates a robust multi-server agent architecture using Python. The system 
features a centra
{'producer': 'Skia/PDF m140 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '../data/pdf/mcp project.pdf', 'file_path': '../data/pdf/mcp project.pdf', 'total_pages': 7, 'format': 'PDF 1.4', 'title': 'Untitled document', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}


In [28]:
texts=[doc.page_content for doc in docs]
## gnerate embbeding
embeddings=embedding_manager.genrate_embedding(texts)
## store in vectore db
vectorstore.add_documents(docs,embeddings)

Generating embeddings for 15 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.18it/s]


Genrated embeddings with shape:(15, 384)
Adding 15 documents to the vector store...
Successfully added 15 documents to the vector store
Total documents in collection:15


## Retriever Pipeline from VectorStore

In [52]:
class RAGRetriever:
    """Retriever pipeline to fetch relevant documents from vector store based on query"""
    def __init__(self,vector_store:VectorStore,embedding_manager:EmbeddingManager):
        """initialize the retriever
        ARGS:
            vector_store: Instance of VectorStore
            embedding_manager: Instance of EmbeddingManager
            top_k: Number of top similar documents to retrieve
        """
        self.vector_store=vector_store
        self.embedding_manager=embedding_manager
    
    
    def retrieve(self,query:str,top_k: int=5,score_threshold: float=0.0)-> List[Dict[str,Any]]:
        """retrieve relevant documents for a given query
        ARGS:
            query: User query string
        
        RETURNS:
            List of retrieved documents with metadata"""
        print(f"Retrieving top {top_k} documents for query: {query}")
        print(f"to_ k: {top_k}, score_threshold: {score_threshold}")
        
        # Step 1: Generate embedding for the query
        query_embedding=self.embedding_manager.genrate_embedding([query])[0] # single query
        
        # Step 2: Query the vector store for similar documents
        try:
            results=self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
             
            )
      
        
        
        # Step 3: Format and return results
            retrieved_doc=[]
       
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0] 
                distances=results['distances'][0]
                ids=results['ids'][0]

                for i,(doc,meta,dist,idss)in enumerate(zip(documents,metadatas,distances,ids)):
                    ## convert distance to similarity score
                    similarity_score=1-(float(dist)/2)  # assuming dist is in [0,1]

                    if similarity_score >=score_threshold:
                        retrieved_doc.append({
                            "id":idss,
                            "content":doc,
                            "metadata":meta,
                            "similarity_score":similarity_score,
                            'distance':float(dist),
                            'rank':i+1
                        })
                print(f"Retrieved {len(retrieved_doc)} documents after applying score threshold")
            else:
                print("No documents found in the vector store.")

            return retrieved_doc
        except Exception as e:
            print(f"Error retrieving documents: {e}")
            return []
        
rag_retriever=RAGRetriever(vectorstore,embedding_manager)
           

In [30]:
rag_retriever

In [53]:
rag_retriever.retrieve("project overview")

Retrieving top 5 documents for query: project overview
to_ k: 5, score_threshold: 0.0
Generating embeddings for 1 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00, 20.20it/s]

Genrated embeddings with shape:(1, 384)
Retrieved 5 documents after applying score threshold


[{'id': 'doc_2b9c89d81',
  'content': 'powerful and scalable AI agents. \n2. Features \n●\u200b Multi-Server Connectivity: Connects to multiple tool servers simultaneously using \nlangchain_mcp_adapters. \n●\u200b Hybrid Transports: Demonstrates the use of both stdio for local subprocess \ncommunication and streamable_http for network-based communication. \n●\u200b Live Data Integration: The weather server calls a real-world API (WeatherAPI.com) \nto provide live data. \n●\u200b Intelligent Agent: Uses a Groq LLM and a LangGraph agent to understand natural \nlanguage prompts and execute the correct tool calls. \n●\u200b Secure Credential Management: API keys and other secrets are managed securely \nusing environment variables and a .env file. \n3. Project Structure \nYour project folder should look like this: \n. \n├── .env \n├── .venv/ \n├── client.py',
  'metadata': {'source': '../data/pdf/mcp project.pdf',
   'page': 0,
   'file_path': '../data/pdf/mcp project.pdf',
   'producer': '

## integration vector db context pipeline with llm output

In [ ]:
## Simple Rag pipeline witth groq llm
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv() 

## INITIALIZE GROQ LLM
groq_api_key="put api key here"
llm=ChatGroq(groq_api_key=groq_api_key,model_name="gemma2-9b-it",temperature=0.1,max_tokens=1024)

## Simple rag function : retreive context +genrate response
def rag_simple(query,retriever,llm,top=3):
    ## retrieve context
    results=retriever.retrieve(query,top_k=top)
    context="\n\n".join([doc["content"]for doc in results])if results else""
    if not context:
        return "No relevant context found "
    
    ## genrate response
    prompt="""use the following context to answer the question concisely.
        {context}

        question:{query}
       
        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content
        



In [56]:
answer=rag_simple("project overview",rag_retriever,llm,top=3)
print(answer)

Retrieving top 3 documents for query: project overview
to_ k: 3, score_threshold: 0.0
Generating embeddings for 1 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.82it/s]

Genrated embeddings with shape:(1, 384)


Retrieved 3 documents after applying score threshold
This project demonstrates a multi-server AI agent architecture using Python. It features a central client that interacts with two specialized tool servers: a Math Server for arithmetic operations and a Weather Server for fetching live weather data.  The client uses a Groq LLM and a LangGraph agent to determine which tool to use based on user prompts. 

